In [42]:
from datetime import datetime

In [ ]:
start = datetime(2000, 1, 1)
end = datetime(2025, 1, 1)

In [ ]:
data_dir = "DATA_DIRECTORY"

# Model drivers

In [85]:
import numpy as np
import pandas as pd

from swvo.io.omni import OMNIHighRes
from swvo.io.sme import SMESuperMAG
from swvo.io.hp import Hp30GFZ

from scipy.interpolate import PchipInterpolator

In [88]:
# OMNI
OMNI_DIR = f"{data_dir}/OMNI SWVO"
omni_loader = OMNIHighRes(OMNI_DIR)
omni = omni_loader.read(start, end, download=True)
omni = omni.drop(columns="file_name")
omni.index.name = "Time tag"

# SME
SME_DIR = f"{data_dir}/SME SWVO"
sme_loader = SMESuperMAG("dmitrii_gurev", SME_DIR)
sme = sme_loader.read(start, end, download=True)
sme = sme.drop(columns="file_name")
sme.index.name = "Time tag"

# Hp30
HP_DIR = f"{data_dir}/OMNI Hp30"
hp_loader = Hp30GFZ(HP_DIR)
hp30 = hp_loader.read(start, end, download=True)
hp30.index.name = "Time tag"
hp30 = hp30.resample("1min").mean()

# Resample Hp30 to 1 min
inds = hp30.reset_index().index[hp30["hp30"].notna()]
spline = PchipInterpolator(inds, hp30["hp30"].iloc[inds])
hp30.loc[:, "hp30"] = spline(range(len(hp30)))

In [90]:
data = omni.merge(sme, on="Time tag").merge(hp30, on="Time tag")

In [92]:
data.astype(np.float32).to_hdf(f"{data_dir}/drivers.h5", key="df", format="table")

# Cluster data

In [9]:
import os
import shutil
import pandas as pd
import numpy as np
from datetime import date, datetime, timedelta
import calendar

from cluster_data_loader import download_dataset

In [10]:
def days_in_year(year):
    return 365 + calendar.isleap(year)

def daterange(start_date: date, end_date: date):
    days = int((end_date - start_date).days)
    for n in range(days):
        yield start_date + timedelta(n)

In [ ]:
products = [
    "CP_PEA_PITCH_SPIN_DPFlux",
    "CP_RAP_PAD_E3DD",
    "CP_RAP_PAD_L3DD",
    "CP_RAP_IES_BG",
    "CP_AUX_POSGSE_1M",
    "CP_AUX_LSTAR",
    ]

In [ ]:
# Download
for product in products:
    for c in [1, 2, 3, 4]:
        for current_date in daterange(start, end):
            start_date = current_date
            end_date = current_date + timedelta(days=1) - timedelta(seconds=1)

            try:
                download_dataset(f"C{c}_{product}",
                                 start_date, end_date,
                                 "CDF", "All",
                                 data_directory=data_dir)
            except Exception as e:
                print(e)

In [ ]:
# Move to dedicated folders
for product in products:
    if not os.path.exists(os.path.join(data_dir, product)):
        os.mkdir(os.path.join(data_dir, product))

    for c in [1, 2, 3, 4]:
        for dir in os.listdir(data_dir):
            if dir.startswith("CSA_Download"):
                for folder in os.listdir(os.path.join(data_dir, dir)):
                    if folder.startswith(f"C{c}_{product}"):
                        for file in os.listdir(os.path.join(data_dir, dir, folder)):
                            shutil.copy(os.path.join(data_dir, dir, folder, file), os.path.join(data_dir, product, file))

for dir in os.listdir(data_dir):
    if dir.startswith("CSA_Download"):
        shutil.rmtree(os.path.join(data_dir, dir))